# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata object
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List available record sets and their @ids
print('Available record sets:')
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}")

    # Display record set name and description
    print(f"  name: {record_set.name}")
    print(f"  description: {record_set.description}")

    # Display fields and their @id for each record set
    print('  Fields:')
    for field in record_set.fields:
        print(f"    - @id: {field.id} | name: {field.name} | type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example: select all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Load all records for a given record set
    records = list(dataset.records(record_set=record_set_id))
    # Convert to DataFrame
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Example: print DataFrame columns for the first record set
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Columns in record set '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** The following code selects a numeric field and categorical field by their `@id` as discovered above.

In [ ]:
import numpy as np

# For demonstration, pick the first record set and a numeric field within it
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]

    # Attempt to automatically pick a numeric field using fields metadata
    numeric_field_id = None
    group_field_id = None

    for field in next(rs for rs in metadata.record_sets if rs.id == rs_id).fields:
        # Choose the first numeric field
        if field.data_type in ('Number', 'Float', 'Integer') and field.id in df.columns:
            numeric_field_id = field.id
            break

    # Attempt to pick a categorical field for grouping
    for field in next(rs for rs in metadata.record_sets if rs.id == rs_id).fields:
        # Use the first Text or non-numeric field
        if field.data_type in ('Text',) and field.id in df.columns:
            group_field_id = field.id
            break

    if numeric_field_id:
        print(f"Using numeric field '@id': {numeric_field_id}")
        print(f"Using group field '@id': {group_field_id}")

        # Drop missing values to avoid errors in subsequent operations
        df_clean = df.dropna(subset=[numeric_field_id])

        # Basic filtering: select records where the numeric field > its median
        threshold = df_clean[numeric_field_id].median()
        filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by group_field and get mean of the numeric field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by '{group_field_id}': Mean of '{numeric_field_id}':")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: histogram and boxplot of the numeric field, colored by group field if available
if record_set_ids and numeric_field_id and rs_id in dataframes:
    plt.figure(figsize=(8,4))
    sns.histplot(data=df_clean, x=numeric_field_id, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if group_field_id and group_field_id in df_clean.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_clean)
        plt.title(f"Distribution of {numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR² dataset on EV protein analysis using `mlcroissant`. We've programmatically listed record sets and fields, loaded data into pandas DataFrames, and performed basic filtering, normalization, grouping, and visualization. Use the `@id` references for robust access to specific data entities in reproducible pipelines.

**Next steps:**
- Further domain-specific analyses, such as statistical tests, modeling, or protein function annotation.
- Exporting clean tables for downstream bioinformatics pipelines.
- Integrating with additional mass spectrometry or proteomic tools.